# Simple Campus Menu Agent

This notebook creates a simple agent that can answer questions about today's campus menu.
It uses the `get_menu` tool from `getmenus.py` to look up data from the database.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import display, Markdown
from loguru import logger

# Import the get_menu tool and helper functions from getmenus.py
from getmenus import get_menu, get_today, setup_logging

setup_logging()
load_dotenv()
print("Environment loaded.")

c:\Users\Zulfiyya\Desktop\hai5016-project\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-06-06 19:51:18.597 | INFO     | getmenus:setup_logging:47 - Logging to C:\Users\Zulfiyya\Desktop\hai5016-project\logs\menu_agent_2026-06-06.log


Environment loaded.


In [2]:
logger.info(f"Today's date (Seoul time): {get_today()}")

2026-06-06 19:51:18.608 | INFO     | __main__:<module>:1 - Today's date (Seoul time): 2026-06-06


## Create the Language Model

Connect to Azure OpenAI using credentials stored in `.env`.

In [3]:
# Create the language model using Azure OpenAI credentials from the .env file
llm = ChatOpenAI(
    base_url=os.environ["AZURE_FOUNDRY_ENDPOINT"],
    api_key=os.environ["AZURE_FOUNDRY_API_KEY"],
    model=os.environ["AZURE_FOUNDRY_MODEL"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

logger.info("LLM ready.")


2026-06-06 19:51:19.139 | INFO     | __main__:<module>:11 - LLM ready.


## Build the Agent

Create the agent with the `get_menu` tool and a helpful system prompt.

In [4]:
today = get_today()

# The system prompt tells the agent how to behave and what today's date is
SYSTEM_PROMPT = (
    f"You are a helpful campus menu assistant. Today's date is {today} (Asia/Seoul). "
    "Use the get_menu tool to fetch menu data for the requested date. "
    "Translate all Korean names (universities, restaurants, dishes) to English. "
    "Group the answer by university, then by restaurant. "
    "Include meal type, dish names, serving time, and price when available."
)

# Build the agent with the get_menu tool attached
agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_menu]
)

logger.info("Agent ready.")

2026-06-06 19:51:19.157 | INFO     | __main__:<module>:19 - Agent ready.


## Ask the Agent a Question

Change the `question` variable below and run the cell to get an answer.

In [5]:
# Change this question to ask anything about the menu
question = "What is on the menu today?"

In [6]:
logger.info("Question: {!r}", question)

# Run the agent with our question
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# Extract the final text reply from the agent's response messages
answer = ""
for message in reversed(result.get("messages", [])):
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        answer = content
        break

logger.info("Answer: {} chars", len(answer))

# Display the answer nicely formatted in the notebook
display(Markdown(answer))

2026-06-06 19:51:19.170 | INFO     | __main__:<module>:1 - Question: 'What is on the menu today?'
2026-06-06 19:51:22.230 | INFO     | getmenus:get_menu:104 - get_menu returned 17 row(s) for 2026-06-06
2026-06-06 19:51:37.956 | INFO     | __main__:<module>:14 - Answer: 1536 chars


Menus for 2026-06-06

Hanyang University — ERICA Campus
- Student Cafeteria
  - Breakfast (08:30 ~ 09:40) — Price: not listed
    - Fish Cake Skewers and Broth
    - Stir-fried Chicken & Kimchi
    - Beef and Green Onion Hangover Soup
    - Toast / Mocha Bun (Brunch Set)
  - Lunch (11:30 ~ 13:30) — Price: not listed
    - Gangneung Jang Kalguksu (spicy noodle soup)
    - Saucy Tteokbokki (Rice Cake Stew) [Snack Set]
    - Budae-style Stir-fry on Rice (stir-fried sausage on rice)
    - Bibim Naengmyeon (spicy cold buckwheat noodles) & Soy Sauce Bulgogi
    - Aqua Donkatsu (pork cutlet with oriental sauce)
    - Pad Kra Pao with Pork
    - Seafood Yaki Udon
    - Smoked Duck Fried Rice with Black Bean Sauce

Hongik University
- Staff Cafeteria
  - Lunch (11:30 ~ 14:00) — Price: 9,000 KRW
    - Yubu Jangguk (tofu pouch soup), Vegetable Fried Rice, Cheese Hamburg Steak, Tomato Penne Pasta
- Dormitory 2 Student Cafeteria
  - Lunch (11:30 ~ 14:00) — Price: 4,800 KRW
    - White Rice, Bean Sprout Soup, Spicy Braised Chicken, Vegetable Spring Roll & Chili Sauce

Seoul National University (SNU)
- Our Home (Building 901)
  - Breakfast (08:00 ~ 09:30) — Semi-Western Buffet — Price: 5,000 KRW
  - Lunch (11:30 ~ 13:30) — Charcoal-grilled Spicy Pork with Bean Sprouts on Rice & Vegetable Spring Roll — Price: 6,000 KRW
  - Dinner (17:30 ~ 19:30) — Tuna Kimchi Stew (with ramen noodles & steamed egg) — Price: 6,000 KRW

If you want, I can: filter by university, show only vegetarian options, or open the source links for any item.